In [4]:
import glob
import pandas as pd
import sqlite3

In [9]:
conn = sqlite3.connect(":memory:")
files = glob.glob("player_stats/*.csv")

for file in files:
    df = pd.read_csv(file)
    year = file.split("_")[-1].split(".")[0]
    df.to_sql(f"stats_{year}", conn, index=False, if_exists="replace")

years = sorted([file.split("_")[-1].split(".")[0] for file in files])

cols_to_drop = ["special_teams_tds", "headshot_url", "position_group"]

# get columns from one table (they should match across years)
first_year = years[0]
cols = pd.read_sql(f"PRAGMA table_info(stats_{first_year});", conn)
cols = [c[1] for c in cols.values]

keep_cols = [c for c in cols if c not in cols_to_drop]

query = " UNION ALL ".join([
    f"SELECT {', '.join(keep_cols)} FROM stats_{year}"
    for year in years
])

final_query = f"""
CREATE TABLE all_players_cleaned AS
{query}
"""

conn.execute(final_query)